# STEP 3 - Can this attack work at all? (decisive test)

Step 2: XLM-R tokenization **does** change under attack, but evasion is only 1.6% at budget 5.
Either the attack is under-powered, or XLM-R is genuinely robust. This notebook decides.

Three suspicions, tested in order:
1. **Position scarcity** - the strict Lo+Lo rule may allow only 1-2 insertions per comment,
   so "budget=5" never actually inserts 5.
2. **Random targeting** - inserting at random ignores which word carries the toxicity.
3. **Real robustness** - XLM-R may simply not care.

We push the attack to its ceiling. If a maxed-out, rendering-unsafe, importance-targeted
attack still fails, the security framing is dead and we pivot with confidence.

Settings: **GPU T4 x2, Internet ON, dataset `naurosromim/bdshs`**. ~25 min.


In [ ]:
# 1. Setup
!pip install -q "emoji==1.7.0" regex ftfy
!pip install -q git+https://github.com/csebuetnlp/normalizer
import os,glob,json,random,unicodedata,warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
from normalizer import normalize
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV="cuda"; ZWNJ,ZWJ,ZWSP="\u200c","\u200d","\u200b"; HASANT="\u09cd"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Three position rules: safe -> semi -> unsafe upper bound
def is_bn(c): return 0x0980<=ord(c)<=0x09FF

def pos_strict(t):      # both neighbours base letters -> rendering guaranteed identical
    return [i for i in range(1,len(t))
            if is_bn(t[i-1]) and is_bn(t[i])
            and unicodedata.category(t[i-1])=="Lo" and unicodedata.category(t[i])=="Lo"]

def pos_semi(t):        # anywhere between Bangla chars, just not touching hasant
    return [i for i in range(1,len(t))
            if is_bn(t[i-1]) and is_bn(t[i]) and t[i-1]!=HASANT and t[i]!=HASANT]

def pos_any(t):         # UPPER BOUND - rendering may change, not a usable attack
    return [i for i in range(1,len(t)) if is_bn(t[i-1]) and is_bn(t[i])]

RULES={"strict":pos_strict,"semi":pos_semi,"any_unsafe":pos_any}

def attack(t,budget=1,char=ZWNJ,seed=0,rule="strict"):
    p=RULES[rule](t)
    if not p: return t,0
    r=random.Random(seed)
    k=len(p) if budget=="ALL" else min(budget,len(p))
    ch=sorted(r.sample(p,k),reverse=True)
    for i in ch: t=t[:i]+char+t[i:]
    return t,k

In [ ]:
# 3. Data
paths=glob.glob("/kaggle/input/**/*.csv",recursive=True)
pick=lambda k:[p for p in paths if k in os.path.basename(p).lower()][0]
train=pd.read_csv(pick("train")); test=pd.read_csv(pick("test"))
obj=[c for c in train.columns if train[c].dtype==object]
TEXT=max(obj,key=lambda c:train[c].astype(str).str.len().mean())
LABEL=[c for c in train.columns if c!=TEXT and train[c].dropna().nunique()==2][0]
def prep(d):
    d=d[[TEXT,LABEL]].dropna().copy(); d.columns=["text","label"]
    d["text"]=d["text"].astype(str); d["label"]=d["label"].astype(int)
    return d[d["text"].str.strip().str.len()>0].reset_index(drop=True)
train,test=prep(train),prep(test)
train=train.sample(20000,random_state=SEED).reset_index(drop=True)
HATE=1

# DIAGNOSTIC: how many insertion slots actually exist?
diag=[]
for name,fn in RULES.items():
    c=[len(fn(t)) for t in test["text"].head(3000)]
    diag.append({"rule":name,"mean_slots":round(np.mean(c),2),
                 "median":int(np.median(c)),"pct_with_0":round(float(np.mean(np.array(c)==0)),4),
                 "pct_lt_5":round(float(np.mean(np.array(c)<5)),4)})
diag=pd.DataFrame(diag); print(diag.to_string(index=False))

In [ ]:
# 4. Train XLM-R (the only vulnerable trained model)
M="xlm-roberta-base"; MAXLEN=128
tk=AutoTokenizer.from_pretrained(M)
mdl=AutoModelForSequenceClassification.from_pretrained(M,num_labels=2).to(DEV)
enc=lambda ts: tk([normalize(t) for t in ts],truncation=True,padding="max_length",
                  max_length=MAXLEN,return_tensors="pt")
e=enc(train["text"].tolist()); y=torch.tensor(train["label"].values)
dl=DataLoader(TensorDataset(e["input_ids"],e["attention_mask"],y),batch_size=32,shuffle=True)
opt=torch.optim.AdamW(mdl.parameters(),lr=2e-5)
sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=2e-5,total_steps=2*len(dl),pct_start=0.1)
sc=torch.cuda.amp.GradScaler(); mdl.train()
for ep in range(2):
    for st,(i_,m_,l_) in enumerate(dl):
        i_,m_,l_=i_.to(DEV),m_.to(DEV),l_.to(DEV); opt.zero_grad()
        with torch.cuda.amp.autocast(): loss=mdl(input_ids=i_,attention_mask=m_,labels=l_).loss
        sc.scale(loss).backward(); sc.step(opt); sc.update(); sch.step()
        if st%200==0: print(f"ep{ep} {st}/{len(dl)} {loss.item():.4f}")
mdl.eval()

@torch.no_grad()
def logits(ts,bs=128):
    o=[]
    for i in range(0,len(ts),bs):
        b=enc(ts[i:i+bs])
        with torch.cuda.amp.autocast():
            o.append(mdl(input_ids=b["input_ids"].to(DEV),
                         attention_mask=b["attention_mask"].to(DEV)).logits.float().cpu())
    return torch.cat(o)
pred=lambda ts: logits(ts).argmax(-1).numpy()

tt=test["text"].tolist(); ty=test["label"].values
cp=pred(tt); acc=accuracy_score(ty,cp); f1=f1_score(ty,cp)
hit=np.where((ty==HATE)&(cp==HATE))[0][:1200]
sub=[tt[i] for i in hit]
print(f"clean acc {acc:.4f} f1 {f1:.4f} | targets {len(sub)}")

In [ ]:
# 5. ATTACK LADDER - random insertion, all rules, up to ALL slots
lad=[]
for rule in RULES:
    for bud in [1,3,5,10,"ALL"]:
        atk,ks=[],[]
        for j,t in enumerate(sub):
            a,k=attack(t,budget=bud,char=ZWNJ,seed=SEED+j,rule=rule); atk.append(a); ks.append(k)
        ev=float((pred(atk)!=HATE).mean())
        lad.append({"rule":rule,"budget":str(bud),"mean_inserted":round(float(np.mean(ks)),2),
                    "evasion_rate":round(ev,4)})
        print(f"{rule:11} budget={str(bud):4} actually_inserted={np.mean(ks):5.2f} evasion {ev:.2%}")
lad=pd.DataFrame(lad); lad.to_csv("/kaggle/working/step3_ladder.csv",index=False)

In [ ]:
# 6. TARGETED attack - occlusion finds the toxic word, then flood it
TGT=400
tsub=sub[:TGT]
def most_important(t):
    w=t.split()
    if len(w)<2: return 0
    base=logits([t])[0]; margin=(base[HATE]-base[1-HATE]).item()
    cands=[" ".join(w[:i]+w[i+1:]) for i in range(len(w))]
    lg=logits(cands)
    drops=[margin-(lg[i][HATE]-lg[i][1-HATE]).item() for i in range(len(w))]
    return int(np.argmax(drops))

tres=[]
for rule in ["strict","semi","any_unsafe"]:
    atk=[]
    for t in tsub:
        w=t.split(); k=most_important(t)
        a,_=attack(w[k],budget="ALL",char=ZWNJ,seed=SEED,rule=rule)
        w[k]=a; atk.append(" ".join(w))
    ev=float((pred(atk)!=HATE).mean())
    tres.append({"attack":"targeted_flood","rule":rule,"evasion_rate":round(ev,4)})
    print(f"targeted+{rule:11} evasion {ev:.2%}")

# and: flood EVERY word (absolute ceiling)
for rule in ["semi","any_unsafe"]:
    atk=[" ".join(attack(w,budget="ALL",char=ZWNJ,seed=SEED,rule=rule)[0] for w in t.split())
         for t in tsub]
    ev=float((pred(atk)!=HATE).mean())
    tres.append({"attack":"flood_all_words","rule":rule,"evasion_rate":round(ev,4)})
    print(f"flood_all+{rule:11} evasion {ev:.2%}")
tres=pd.DataFrame(tres); tres.to_csv("/kaggle/working/step3_targeted.csv",index=False)

In [ ]:
# 7. Mechanism: how much does tokenization actually inflate?
fert=[]
for rule in RULES:
    a=[attack(t,budget="ALL",char=ZWNJ,seed=SEED,rule=rule)[0] for t in sub[:500]]
    cl=[len(tk.tokenize(normalize(t))) for t in sub[:500]]
    at=[len(tk.tokenize(normalize(t))) for t in a]
    fert.append({"rule":rule,"mean_tokens_clean":round(float(np.mean(cl)),2),
                 "mean_tokens_attacked":round(float(np.mean(at)),2),
                 "inflation_pct":round(100*(np.mean(at)/np.mean(cl)-1),1)})
fert=pd.DataFrame(fert); print(fert.to_string(index=False))
fert.to_csv("/kaggle/working/step3_fertility.csv",index=False)

In [ ]:
# 8. SUMMARY  <-- copy back
print("="*64); print("STEP 3 SUMMARY  (xlm-roberta-base)"); print("="*64)
print(f"clean acc/f1: {acc:.4f} / {f1:.4f} | targets {len(sub)}")
print("\n-- insertion slots available --"); print(diag.to_string(index=False))
print("\n-- random attack ladder --");      print(lad.to_string(index=False))
print("\n-- targeted / flood --");          print(tres.to_string(index=False))
print("\n-- tokenization inflation --");    print(fert.to_string(index=False))
print("="*64)